# Multi-Agent Communication & Negotiation with LangChain

This notebook demonstrates a **multi-agent negotiation system** for product feature prioritization. Three specialist agents — each representing a different stakeholder — evaluate features from their own perspective. A coordinating LLM calls each agent as a tool, collects the scores, and synthesizes a final prioritized ranking.

**What you'll learn:**
- How to define Python functions as callable LangChain tools
- How to bind tools to a chat model with `bind_tools`
- How to build an agentic loop that handles tool calls and routes results back to the model
- How structured tool use enables multi-agent "negotiation" from a single orchestrating LLM

**Scenario:** A product team needs to prioritize three features — *Dark Mode*, *AI-based Smart Search*, and *Offline Support* — by gathering input from a User Advocate, a Tech Lead, and a Business Analyst.

## Step 1: Install Dependencies

Run this cell once to install the required packages. You can skip it if they're already installed.

| Package | Purpose |
|---|---|
| `langchain` | Core LangChain framework — chains, agents, message types |
| `langchain-openai` | OpenAI integration: `ChatOpenAI`, tool binding, embeddings |
| `langchain-community` | Community-contributed tools and integrations |
| `python-dotenv` | Loads API keys from a `.env` file so they stay out of your code |

In [ ]:
%pip install -U langchain langchain-openai langchain-community dotenv

## Step 2: Import Dependencies

| Import | Purpose |
|---|---|
| `os` | Read environment variables at runtime |
| `load_dotenv` | Parse and inject `.env` file contents into the process environment |
| `ChatOpenAI` | LangChain wrapper around OpenAI's chat completion API |
| `Tool` | Wraps a Python function so an LLM can invoke it as a named tool |
| `HumanMessage` | Represents a user turn in the conversation history sent to the model |
| `ToolMessage` | Carries the return value of a tool call back into the conversation |

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_core.messages import HumanMessage, ToolMessage

## Step 3: Load Environment Variables

`load_dotenv()` reads your `.env` file and injects its contents into `os.environ`. This keeps sensitive values like your API key out of the notebook source.

Create a `.env` file in this project directory with the following content:

```
OPENAI_API_KEY=sk-...
```

In [3]:
load_dotenv()

True

## Step 4: Define the Scoring Functions

Each function below simulates a **stakeholder perspective** on a product feature. They look up a feature name in a hardcoded dictionary and return a score (1–10) plus a short justification.

In a real system, these functions might call APIs, query databases, or run ML models. For this walkthrough, the static data keeps things simple so we can focus on the agent mechanics.

We'll define three scorers:
1. **User Advocate** — how much end users want the feature
2. **Tech Lead** — how difficult it is to build
3. **Business Analyst** — how much business value it delivers

### User Advocate

Scores features by **user desirability** — how much end users want or need them, based on feedback, support requests, and user research.

In [4]:
def user_advocate_tool(feature):
    scores = {
        "Dark Mode": (5, "Some users want it for comfort"),
        "AI-based Smart Search": (9, "Highly requested feature"),
        "Offline Support": (7, "Useful for travel and remote work")
        }
    score, comment = scores.get(feature, (0, "No data"))
    return f"{feature} → User Score: {score}. Note: {comment}"

### Tech Lead

Scores features by **technical feasibility** — implementation complexity, required infrastructure, and engineering effort. Higher scores mean easier to build.

In [5]:
def tech_lead_tool(feature):
    scores = {
        "Dark Mode": (8, "Easy to implement"),
        "AI-based Smart Search": (5, "Requires custom model integration"),
        "Offline Support": (6, "Moderate complexity")
        }
    score, comment = scores.get(feature, (0, "No data"))
    return f"{feature} → Tech Score: {score}. Note: {comment}"

### Business Analyst

Scores features by **business value** — expected impact on revenue, customer retention, and strategic goals.

In [6]:
def business_analyst_tool(feature):
    scores = {
        "Dark Mode": (4, "Minimal impact on revenue"),
        "AI-based Smart Search": (9, "Improves retention and engagement"),
        "Offline Support": (6, "Moderate business value")
        }
    score, comment = scores.get(feature, (0, "No data"))
    return f"{feature} → Business Score: {score}. Note: {comment}"

## Step 5: Register Functions as LangChain Tools

The `Tool` wrapper exposes each Python function to the LLM as a callable tool. Three fields matter:

- **`name`** — how the model refers to the tool in its response (must be unique)
- **`func`** — the Python function to call when the tool is invoked
- **`description`** — a natural-language hint the model uses to decide *when* to call this tool

The descriptions are especially important: the model reads them at inference time to select the right tool for each task. Vague descriptions lead to wrong tool choices.

In [7]:
tools = [
    Tool(name="UserAdvocate", func=user_advocate_tool, description="User desirability scoring"),
    Tool(name="TechLead", func=tech_lead_tool, description="Technical feasibility scoring"),
    Tool(name="BusinessAnalyst", func=business_analyst_tool, description="Business value scoring")
    ]

## Step 6: Initialize the Model and Bind Tools

`ChatOpenAI` connects to the OpenAI API using the key we loaded from `.env`.

Calling `.bind_tools(tools)` attaches the tool definitions to every request made with `llm_with_tools`. This tells the model:
- What tools exist (by name and description)
- What arguments each tool accepts

From this point on, the model can choose to call any of the three scoring tools instead of — or before — generating a final text response.

In [8]:
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model_name="gpt-5-mini"
)

llm_with_tools = llm.bind_tools(tools)

## Step 7: Build the Agentic Loop

`run_agent` implements the core **tool-use loop** that turns an LLM into an agent:

1. **Send** the current message history to the model
2. **Check** — did the model request any tool calls?
   - **Yes** → execute each tool and append the results as `ToolMessage` objects, then go back to step 1
   - **No** → the model has produced its final response; return it

This loop is what makes an LLM *agentic* — it keeps running, calling tools and incorporating results, until the model decides it has enough information to give a final answer.

> **Note:** The `__arg1` check handles a quirk in how some models pass single-argument tool calls. If a tool only takes one argument, certain model versions wrap it under the key `"__arg1"` rather than using the parameter's actual name.

In [9]:
def run_agent(prompt: str):
    messages = [HumanMessage(content=prompt)]
    
    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        
        if not response.tool_calls:
            return response.content
        
        for tool_call in response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call.get("args", {})
            
            selected_tool = next(t for t in tools if t.name == tool_name)
        
            if "__arg1" in tool_args:
                tool_output = selected_tool.func(tool_args["__arg1"])
            else:
                tool_output = selected_tool.func(**tool_args)
            
            messages.append(
                ToolMessage(
                    content=str(tool_output),
                    tool_call_id=tool_call["id"],
                )
            )

## Step 8: Write the Agent Prompt

The prompt is the instruction we give the agent. It tells the model:
- **What to evaluate** — the three features
- **Which tools to use** — all three scoring tools, for each feature
- **What to produce** — a prioritized list

Writing a clear, structured prompt is critical for agentic tasks. The model uses this instruction to decide which tools to call, with which arguments, and in what order. In this case we expect **9 tool calls** in total (3 features × 3 tools).

In [10]:
prompt = """
Evaluate each of the following features one by one:
Dark Mode, AI-based Smart Search, and Offline Support.

Use the available tools to get:
- User Score (from UserAdvocate)
- Technical Feasibility Score (from TechLead)
- Business Value Score (from BusinessAnalyst)
Then combine the scores and return a prioritized list from highest to lowest.
"""

## Step 9: Run the Agent

Pass the prompt to `run_agent` and print the final result. Watch what happens behind the scenes:

1. The model reads the prompt and starts calling tools — first `UserAdvocate("Dark Mode")`, then `TechLead("Dark Mode")`, and so on
2. After all 9 tool calls complete, the model has all the scores it needs
3. It synthesizes the results and returns a ranked feature list

The multi-agent "negotiation" has occurred entirely through structured tool calls — no separate agent frameworks or message queues needed.

In [11]:
result = run_agent(prompt)
print(result)

Here are the results from the three evaluators (UserAdvocate, TechLead, BusinessAnalyst), followed by combined scoring and a prioritized list.

1) Dark Mode
- User Score: 5 — Note: Some users want it for comfort
- Technical Feasibility Score: 8 — Note: Easy to implement
- Business Value Score: 4 — Note: Minimal impact on revenue
- Combined score (sum): 17
- Quick take: Low technical risk and quick win, but limited business upside and only moderate user demand.

2) AI-based Smart Search
- User Score: 9 — Note: Highly requested feature
- Technical Feasibility Score: 5 — Note: Requires custom model integration
- Business Value Score: 9 — Note: Improves retention and engagement
- Combined score (sum): 23
- Quick take: Highest user and business impact but non-trivial engineering effort and integration complexity. High priority for strategic value.

3) Offline Support
- User Score: 7 — Note: Useful for travel and remote work
- Technical Feasibility Score: 6 — Note: Moderate complexity
- Busi